# TP4 : Évaluation du RAG

**Contexte :** assistant du service client de Lagune Banque.

Un bon chunk ne garantit pas une bonne réponse : le modèle peut ajouter ce qui n'y figure pas, ou répondre à côté. Vous allez mesurer la réponse avec les quatre métriques de RAGAS, dont vous écrivez deux, puis comparer trois versions de l'assistant : **départ** (recherche vectorielle, réponse libre), **avec reranker** (TP2) et **sous contrôle** (refus et citations vérifiées du TP3).

Complétez les `# TODO`.

## <a id="syntaxe"></a>Syntaxe

Le modèle sert de juge : il note une réponse ou des chunks, en sortie structurée. Un schéma Pydantic décrit la forme attendue :

```python
class Weather(BaseModel):
    city: str
    days: list[int]
```

`ra.structured(prompt, Schema)` envoie le prompt au modèle avec ce schéma et renvoie l'objet validé par Pydantic, ou `None` si le modèle ne respecte pas le schéma.

## <a id="configuration"></a>Configuration

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage")] if (p / "rag.py").exists()).resolve()))
import rag
import rag_advanced as ra
from config import prepare
from weaviate_store import connect

In [ ]:
prepare(embedding=True, rerank=True)

## <a id="versions"></a>Versions de l'assistant

Chaque version prend une question et renvoie un dict : `reponse`, `refus` et les `chunks` utilisés. L'index est celui du TP1.

Le prompt des versions 1 et 2 : répondre seulement à partir des chunks, citer la source, refuser hors corpus. `rag.build_prompt` y insère les chunks et la question.

In [ ]:
PROMPT_RAG = """Tu es l'assistant du service client de Lagune Banque.

# Règle 1 - Périmètre strict
Réponds UNIQUEMENT à partir des chunks fournis ci-dessous.
Si l'information n'y figure pas, réponds exactement :
"Je ne trouve pas cette information dans la documentation fournie."
N'invente jamais un chiffre, un délai ou une condition.

# Règle 2 - Citation
Pour chaque affirmation, indique la source entre crochets, par exemple [DOC-06].
Si plusieurs chunks se contredisent, signale-le explicitement.

# Règle 3 - Concision
Réponds en 2 à 4 phrases maximum, style professionnel et direct.

# Chunks fournis
{chunks}

# Question du client
{question}
"""

Le prompt de la version 3, celui du TP3 : le modèle renvoie une réponse et des extraits copiés mot pour mot, que `ra.answer_with_sources` vérifie ensuite dans les chunks.

In [ ]:
PROMPT_SOURCES = """Tu es l'assistant du service client de Lagune Banque.
Réponds UNIQUEMENT à partir des chunks fournis, en 2 à 4 phrases. N'invente aucun chiffre, délai ou condition.
Si l'information n'y figure pas, mets "reponse" à "REFUS" et laisse "citations" vide.
Pour chaque citation, "source" est l'identifiant seul (DOC-xx, sans la version) et "extrait" un texte copié mot pour mot depuis le chunk.

# Chunks
{chunks}

# Question
{question}
"""

In [ ]:
documents = rag.load_corpus()
for d in documents:
    if d["fichier"].endswith(".pdf") and not d["avertissement"]:
        d["pages"] = rag.strip_headers(d["pages"])
index = rag.index_chunks(connect(), "index_mesure", rag.split_documents(documents, 500, 75, "mesure"))


def retrieve(question):
    return rag.rerank(question, rag.search(index, question, 20, rag.AUJOURDHUI), 3)


def answer_plain(question, chunks):
    return {"reponse": rag.generate(rag.build_prompt(question, chunks, prompt=PROMPT_RAG)), "refus": False, "chunks": chunks}


VERSIONS = {
    "1. Départ": lambda q: answer_plain(q, rag.search(index, q, 3, rag.AUJOURDHUI)),
    "2. Avec reranker": lambda q: answer_plain(q, retrieve(q)),
    "3. Sous contrôle": lambda q: ra.answer_with_sources(q, retrieve(q), prompt=PROMPT_SOURCES),
}

## <a id="questions"></a>Questions

Les questions viennent de `data/evaluation/questions_rag.json`. Chacune porte les `elements` qui doivent figurer dans la bonne réponse (« 2 000 fcfa », « 48 heures »), ce qui permet de vérifier une réponse sans écrire une réponse de référence. On prend 6 cas courants et 7 cas difficiles, sans historique de conversation, auxquels s'ajoutent 3 pièges hors corpus : le bon comportement est de refuser.

In [ ]:
courants = ra.load_questions("representatif")[::2]
difficiles = [q for q in ra.load_questions("difficile") if not q["historique"]][::4]
PIEGES = ["Proposez-vous des comptes ou des portefeuilles en cryptomonnaie ?", "Pouvez-vous m'accorder un prêt de 200 000 FCFA dès aujourd'hui ?",
          "Quelle est la météo prévue à Abidjan demain ?"]
jeu = [{"question": q["question"], "elements": q["elements"], "piege": False} for q in courants + difficiles]
jeu += [{"question": q, "elements": [], "piege": True} for q in PIEGES]
len(jeu), sum(x["piege"] for x in jeu)

## <a id="metriques"></a>Métriques

| Métrique | Question posée | Ce qu'elle détecte |
|---|---|---|
| **Fidélité** (faithfulness) | Chaque affirmation de la réponse est-elle appuyée par les chunks ? | le modèle invente |
| **Pertinence de la réponse** (answer relevancy) | La réponse répond-elle à la question posée ? | réponse à côté ou trop vague |
| **Précision du contexte** (context precision) | Les chunks utiles sont-ils classés avant les inutiles ? | mauvais classement des chunks |
| **Rappel du contexte** (context recall) | Les chunks contiennent-ils tout ce qu'il faut pour répondre ? | recherche incomplète |

```mermaid
flowchart LR
    Q["Question"]
    C["Chunks retrouvés"]
    R["Réponse"]
    E["Éléments attendus"]
    R ---|"Fidélité"| C
    R ---|"Pertinence"| Q
    C ---|"Précision"| Q
    C ---|"Rappel"| E
```

Les deux premières jugent la **réponse**, les deux autres jugent la **recherche**. Trois utilisent un modèle comme juge ; le rappel se calcule sans modèle, grâce aux `elements`. On ajoute l'**exactitude** : tous les `elements` figurent-ils dans la réponse ?

Le rappel, la pertinence et l'exactitude sont fournis. Vous écrivez la fidélité et la précision.

In [ ]:
from pydantic import BaseModel


def normalize(texte):
    return ra.normalize(texte)


def context_recall(chunks, elements):
    contexte = normalize(" ".join(p["texte"] for p in chunks))
    return sum(e in contexte for e in elements) / len(elements)


def accuracy(reponse, elements):
    return float(all(e in normalize(reponse) for e in elements))


class GeneratedQuestions(BaseModel):
    questions: list[str]


def answer_relevancy(question, reponse):
    """Le modèle imagine trois questions auxquelles la réponse répondrait ; on mesure leur ressemblance avec la vraie question (cosinus des embeddings)."""
    result = ra.structured(f"Voici une réponse d'un assistant bancaire. Écris 3 questions différentes auxquelles elle répond.\n\nRéponse : {reponse}", GeneratedQuestions)
    imaginees = result.questions if result else []
    if not imaginees:
        return 0.0
    vecteurs = rag.embeddings([question] + imaginees)
    produit = lambda a, b: sum(x * y for x, y in zip(a, b))
    norme = lambda a: produit(a, a) ** 0.5
    return sum(produit(vecteurs[0], v) / (norme(vecteurs[0]) * norme(v)) for v in vecteurs[1:]) / len(imaginees)

**TODO 1 : le schéma de la fidélité.** Le modèle décompose la réponse en affirmations. Écrivez `Affirmation` (`texte` : `str`, `appuyee` : `bool`) et `Faithfulness` (`affirmations` : liste d'`Affirmation`).

In [ ]:
# TODO 1 : class Affirmation(BaseModel) et class Faithfulness(BaseModel)

**TODO 2 : la fidélité.** Le modèle décompose la réponse en affirmations et indique pour chacune si les chunks l'appuient. La fidélité est la part d'affirmations appuyées. Le prompt et l'appel au juge sont fournis : `ra.structured(prompt, schéma)` renvoie un objet validé par Pydantic, ou `None` si le modèle ne respecte pas le schéma. Complétez la ligne qui calcule la fidélité.

In [ ]:
PROMPT_FIDELITE = """Décompose la réponse en affirmations simples. Pour chacune, indique si les chunks l'appuient (appuyee vrai) ou non (faux).
Ce qui n'est pas écrit dans les chunks n'est pas appuyé, même si c'est plausible.

# Chunks
{chunks}

# Réponse
{reponse}"""


def faithfulness(reponse, chunks):
    result = ra.structured(PROMPT_FIDELITE.format(chunks=rag.format_chunks(chunks), reponse=reponse), Faithfulness)
    affirmations = result.affirmations if result else []    # result vaut None si le modèle ne respecte pas le schéma
    if not affirmations:
        return None
    appuyees = [a for a in affirmations if a.appuyee]
    # TODO 2 : renvoyez la fidélité, la part d'affirmations appuyées (entre 0 et 1)
    return None

**TODO 3 : la précision du contexte.** Le modèle indique, pour chaque chunk dans l'ordre, s'il aide à répondre. La précision récompense les chunks utiles placés en tête : c'est la moyenne, sur chaque chunk utile, de la part de chunks utiles parmi ceux qui le précèdent (lui compris). Avec trois chunks `[utile, inutile, utile]`, elle vaut (1/1 + 2/3) / 2 = 0,83 ; avec `[inutile, utile, utile]`, (1/2 + 2/3) / 2 = 0,58. Le prompt et l'appel au juge sont fournis : complétez la ligne qui calcule la précision au rang de chaque chunk utile.

In [ ]:
class UsefulChunks(BaseModel):
    utiles: list[bool]      # une valeur par chunk, dans l'ordre


PROMPT_PRECISION = """Voici une question et des chunks numérotés. Pour chaque chunk, dans l'ordre, indique s'il contient une information utile pour répondre à la question.

# Question
{question}

# Chunks
{chunks}"""


def context_precision(question, chunks):
    numerotes = "\n\n".join(f"[{i}] {p['texte']}" for i, p in enumerate(chunks, 1))
    result = ra.structured(PROMPT_PRECISION.format(question=question, chunks=numerotes), UsefulChunks)
    utiles = result.utiles[:len(chunks)] if result else []
    precisions, nb_utiles = [], 0
    for rang, utile in enumerate(utiles, 1):    # rang 1 = le premier chunk
        if utile:
            nb_utiles += 1    # chunks utiles parmi les `rang` premiers
            # TODO 3 : ajoutez la précision à ce rang : la part de chunks utiles parmi les `rang` premiers
            precisions.append(0)
    return sum(precisions) / len(precisions) if precisions else 0.0

## <a id="evaluation"></a>Évaluation

`evaluate` applique une version à toutes les questions et renvoie une ligne par question. Les métriques de réponse (fidélité, pertinence) ne se calculent que si l'assistant a répondu ; sur un piège, le bon comportement est le refus.

Cette cellule appelle le modèle plusieurs fois par question et par version : comptez quelques minutes.

In [ ]:
def evaluate(version):
    lignes = []
    for x in jeu:
        r = VERSIONS[version](x["question"])
        refus = r["refus"] or rag.REFUS in r["reponse"]
        ligne = {"question": x["question"][:50], "piege": x["piege"], "refus": refus}
        if x["piege"]:
            ligne["refus correct"] = refus
        else:
            ligne["exactitude"] = 0.0 if refus else accuracy(r["reponse"], x["elements"])
            ligne["rappel contexte"] = context_recall(r["chunks"], x["elements"])
            ligne["précision contexte"] = context_precision(x["question"], r["chunks"])
            if not refus:
                ligne["fidélité"] = faithfulness(r["reponse"], r["chunks"])
                ligne["pertinence"] = answer_relevancy(x["question"], r["reponse"])
        lignes.append(ligne)
    return pd.DataFrame(lignes)


detail = {version: evaluate(version) for version in VERSIONS}

## <a id="resultats"></a>Résultats

Les moyennes ignorent les valeurs absentes : la fidélité d'une version qui refuse beaucoup porte donc sur peu de questions. La colonne **réponses** donne ce nombre.

In [ ]:
lignes = {}
for version, d in detail.items():
    metier, pieges = d[~d["piege"]], d[d["piege"]]    # questions métier et pièges
    moyennes = metier[["exactitude", "rappel contexte", "précision contexte", "fidélité", "pertinence"]].mean().round(2)
    lignes[version] = {**moyennes,
                       "réponses": int((~metier["refus"]).sum()),    # questions métier auxquelles la version a répondu
                       # TODO 4 : le nombre de pièges refusés (somme de la colonne "refus" de pieges)
                       "pièges refusés": None}
tableau = pd.DataFrame(lignes).T
tableau

## Contrôle automatique

La dernière cellule affiche ÉCHEC tant que les `# TODO` ne sont pas faits.

In [ ]:
try:
    assert set(Affirmation.model_fields) == {"texte", "appuyee"} and set(Faithfulness.model_fields) == {"affirmations"}, "schémas : Affirmation (texte, appuyee) et Faithfulness (affirmations) (TODO 1)"
    assert faithfulness("Le montant minimum est de 5 000 FCFA.", [{"texte": "Le montant minimum d'un prêt est de 5 000 FCFA.", "meta": {"source": "DOC-02", "version": "1.0"}}]) == 1.0, "faithfulness : une réponse entièrement appuyée vaut 1 (TODO 2)"
    assert faithfulness("Le montant minimum est de 9 999 FCFA.", [{"texte": "Le montant minimum d'un prêt est de 5 000 FCFA.", "meta": {"source": "DOC-02", "version": "1.0"}}]) == 0.0, "faithfulness : une affirmation inventée vaut 0 (TODO 2)"
    utile, inutile = {"texte": "Les frais de retard sont de 2 000 FCFA."}, {"texte": "L'agence ferme à 17 heures."}
    q = "Quels sont les frais de retard ?"
    assert context_precision(q, [utile]) == 1.0, "context_precision : un seul chunk utile vaut 1 (TODO 3)"
    assert context_precision(q, [inutile, utile]) < context_precision(q, [utile, inutile]), "un chunk utile en tête doit donner une meilleure précision (TODO 3)"
    assert tableau is not None and tableau.shape[0] == 3 and {"exactitude", "fidélité", "pièges refusés"} <= set(tableau.columns) and tableau["pièges refusés"].notna().all(), "un tableau de 3 versions avec les colonnes demandées (TODO 4)"
    print("OK")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else 'terminez les # TODO et relancez les cellules.'}")